Getting the Data

In [2]:
import duckdb
import pandas as pd
from pathlib import Path
# Make paths relative to this notebook's location
notebook_dir = Path.cwd()
base_dir = notebook_dir.parents[1]
db_path = str(base_dir / "crsp" / "crsp_dataset.duckdb")
parquet_file = str(base_dir / "crsp" / "CRSPdF20022025.parquet")

conn = duckdb.connect(db_path)

conn.execute(f"""
CREATE TABLE IF NOT EXISTS crsp_table AS
SELECT * FROM read_parquet('{parquet_file}')
""")

crsp_table_df = conn.execute("SELECT * FROM crsp_table").fetchdf()
crsp_table_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35000,11.49000,11.19,11.10000
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89000,9.60000,9.27,9.60000
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25000,3.29000,3.24,3.06000
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05000,1.30000,1.03,1.04970
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90000,24.00000,23.80,23.40000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40802450,92835K10,2023-12-29,93426,53443,1.0,1.0,12510.0,34.07,0.013385,0.013385,94057.0,34.10000,34.50000,34.07,33.79000
40802451,G3323L10,2023-12-29,93427,53445,1.0,1.0,36331.0,190.33,-0.033416,-0.033416,296974.0,190.32001,196.05000,190.12,189.50999
40802452,12503M10,2023-12-29,93429,53447,1.0,1.0,105556.0,178.56,0.004049,0.004049,521913.0,178.56000,179.09000,178.55,177.44000
40802453,78513510,2023-12-29,93434,53427,1.0,1.0,43039.0,0.70,0.166667,0.166667,169185.0,0.70010,0.71495,0.70,0.58500


Cleaning Where Return is Zero

In [3]:
clean_crsp_df = conn.execute("""
SELECT *   
FROM crsp_table_df
WHERE ret IS NOT NULL
AND ret <> 0
""").fetchdf()

clean_crsp_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35000,11.49000,11.19,11.10000
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89000,9.60000,9.27,9.60000
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25000,3.29000,3.24,3.06000
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05000,1.30000,1.03,1.04970
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90000,24.00000,23.80,23.40000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38829126,92835K10,2023-12-29,93426,53443,1.0,1.0,12510.0,34.07,0.013385,0.013385,94057.0,34.10000,34.50000,34.07,33.79000
38829127,G3323L10,2023-12-29,93427,53445,1.0,1.0,36331.0,190.33,-0.033416,-0.033416,296974.0,190.32001,196.05000,190.12,189.50999
38829128,12503M10,2023-12-29,93429,53447,1.0,1.0,105556.0,178.56,0.004049,0.004049,521913.0,178.56000,179.09000,178.55,177.44000
38829129,78513510,2023-12-29,93434,53427,1.0,1.0,43039.0,0.70,0.166667,0.166667,169185.0,0.70010,0.71495,0.70,0.58500


(can add dropping penny stocks and winsorizing daily returns)

Add Daily Market Cap (anchor factor)

In [4]:
market_cap_df = conn.execute("""
    SELECT *, abs(prc) * shrout * 1000 AS market_cap
    FROM clean_crsp_df
    """).fetchdf()

market_cap_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo,market_cap
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35000,11.49000,11.19,11.10000,2.848260e+07
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89000,9.60000,9.27,9.60000,8.188800e+07
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25000,3.29000,3.24,3.06000,6.800300e+07
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05000,1.30000,1.03,1.04970,8.696100e+06
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90000,24.00000,23.80,23.40000,1.854972e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38829126,92835K10,2023-12-29,93426,53443,1.0,1.0,12510.0,34.07,0.013385,0.013385,94057.0,34.10000,34.50000,34.07,33.79000,4.262157e+08
38829127,G3323L10,2023-12-29,93427,53445,1.0,1.0,36331.0,190.33,-0.033416,-0.033416,296974.0,190.32001,196.05000,190.12,189.50999,6.914879e+09
38829128,12503M10,2023-12-29,93429,53447,1.0,1.0,105556.0,178.56,0.004049,0.004049,521913.0,178.56000,179.09000,178.55,177.44000,1.884808e+10
38829129,78513510,2023-12-29,93434,53427,1.0,1.0,43039.0,0.70,0.166667,0.166667,169185.0,0.70010,0.71495,0.70,0.58500,3.012730e+07


Value-Weighted Market Return

In [5]:
vw_market_return_df = conn.execute("""
    WITH lagged AS (
  SELECT
    date,
    permno,
    ret,
    LAG(market_cap) OVER (
      PARTITION BY permno
      ORDER BY date
    ) AS lag_mktcap
  FROM market_cap_df
  WHERE ret IS NOT NULL
    AND market_cap IS NOT NULL
),
weighted AS (
  SELECT
    date,
    permno,
    ret,
    lag_mktcap,
    SUM(lag_mktcap) OVER (PARTITION BY date) AS total_lag_mktcap
  FROM lagged
  WHERE lag_mktcap IS NOT NULL
    AND lag_mktcap > 0
)
SELECT
  date,
  SUM( (lag_mktcap / NULLIF(total_lag_mktcap, 0.0)) * ret ) AS market_factor_return,
  COUNT(*) AS n_stocks
FROM weighted
GROUP BY date
ORDER BY date
    """).fetchdf()

vw_market_return_df

,date,market_factor_return,n_stocks
0,2002-01-03,0.010148,6542
1,2002-01-04,0.006835,6836
2,2002-01-07,-0.006808,6900
3,2002-01-08,-0.002744,6845
4,2002-01-09,-0.004530,6885
...,...,...,...
5531,2023-12-22,0.002357,9135
5532,2023-12-26,0.005264,9170
5533,2023-12-27,0.002044,9117
5534,2023-12-28,-0.000062,9116


Ticker Upload

In [19]:
# Read the ticker CSV into a DataFrame, specifying encoding to avoid UnicodeDecodeError

current_dir = Path.cwd()
ticker_path = str(current_dir / "cusip_CAPIQ_data.csv")

ticker_convert_df = pd.read_csv(
    ticker_path,
    encoding="latin1",          # ISO-8859-1 handles most non-UTF8 characters
    low_memory=False
)
ticker_convert_df

,Count,Cusip,CIQ,Company Name,Ticker,Exchange,Country,Industry,EV/EBITDA TTM,EV/EBITDA NTM,EV,EV/UFCF TTM,EV/EBIT TTM,Price/Book TTM,ROIC,Net Debt/EBITDA
0,1,00030710,IQ262747167,"AAC Holdings, Inc.",OTCPK:AACH,OTCPK,United States,Health Care Providers and Services,0,0,0.00000,0,0,0,-20.9396,NM
1,2,00036020,IQ360206,"New Horizon Kids Quest, Inc.",OTCPK:KIDQ,OTCPK,United States,Diversified Consumer Services,0,0,0.00000,0,0,0,7.9809,0.56115
2,3,00036110,IQ168154,AAR Corp.,NYSE:AIR,NYSE,United States,Aerospace and Defense,14.97109,13.65208,5644.10256,79.21547,20.41267,2.99475,3.8789,2.59775
3,4,00037520,IQ18527,ABB Ltd,SWX:ABBN,SWX,Switzerland,Electrical Equipment,23.58588,21.08793,130842.14130,127.44298,27.91993,10.06755,11.5772,0.37379
4,5,00039J10,IQ1790631728,"AB Active ETFs, Inc. - AB Ultra Short Income ETF",ARCA:YEAR,ARCA,United States,Capital Markets,0,0,0.00000,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4607,4608,Y8162K20,IQ266636382,Star Bulk Carriers Corp.,NasdaqGS:SBLK,NasdaqGS,Greece,Marine Transportation,9.66643,6.7052,3698.52273,0,24.76711,1.22447,2.2184,1.82382
4608,4609,Y8564W10,IQ329438,Teekay Corporation Ltd.,NYSE:TK,NYSE,Bermuda,"Oil, Gas and Consumable Fuels",4.89815,0,1518.66212,10.56813,8.45962,1.67251,3.8572,NM
4609,4610,Y8565J10,IQ30484742,Altera Infrastructure L.P.,NYSE:TOO,NYSE,United Kingdom,"Oil, Gas and Consumable Fuels",0,0,0.00000,0,0,0,-2.5632,6.10463
4610,4611,Y8565N30,IQ39279824,Teekay Tankers Ltd.,NYSE:TNK,NYSE,Bermuda,"Oil, Gas and Consumable Fuels",5.28776,5.38904,1885.15154,87.70594,8.39244,1.32097,14.4688,NM


Ticker Clean

In [20]:
ticker_clean_df = conn.execute("""
    SELECT * EXCLUDE ('Count')
    FROM ticker_convert_df
    """).fetchdf()

ticker_clean_df

,Cusip,CIQ,Company Name,Ticker,Exchange,Country,Industry,EV/EBITDA TTM,EV/EBITDA NTM,EV,EV/UFCF TTM,EV/EBIT TTM,Price/Book TTM,ROIC,Net Debt/EBITDA
0,00030710,IQ262747167,"AAC Holdings, Inc.",OTCPK:AACH,OTCPK,United States,Health Care Providers and Services,0,0,0.00000,0,0,0,-20.9396,NM
1,00036020,IQ360206,"New Horizon Kids Quest, Inc.",OTCPK:KIDQ,OTCPK,United States,Diversified Consumer Services,0,0,0.00000,0,0,0,7.9809,0.56115
2,00036110,IQ168154,AAR Corp.,NYSE:AIR,NYSE,United States,Aerospace and Defense,14.97109,13.65208,5644.10256,79.21547,20.41267,2.99475,3.8789,2.59775
3,00037520,IQ18527,ABB Ltd,SWX:ABBN,SWX,Switzerland,Electrical Equipment,23.58588,21.08793,130842.14130,127.44298,27.91993,10.06755,11.5772,0.37379
4,00039J10,IQ1790631728,"AB Active ETFs, Inc. - AB Ultra Short Income ETF",ARCA:YEAR,ARCA,United States,Capital Markets,0,0,0.00000,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4607,Y8162K20,IQ266636382,Star Bulk Carriers Corp.,NasdaqGS:SBLK,NasdaqGS,Greece,Marine Transportation,9.66643,6.7052,3698.52273,0,24.76711,1.22447,2.2184,1.82382
4608,Y8564W10,IQ329438,Teekay Corporation Ltd.,NYSE:TK,NYSE,Bermuda,"Oil, Gas and Consumable Fuels",4.89815,0,1518.66212,10.56813,8.45962,1.67251,3.8572,NM
4609,Y8565J10,IQ30484742,Altera Infrastructure L.P.,NYSE:TOO,NYSE,United Kingdom,"Oil, Gas and Consumable Fuels",0,0,0.00000,0,0,0,-2.5632,6.10463
4610,Y8565N30,IQ39279824,Teekay Tankers Ltd.,NYSE:TNK,NYSE,Bermuda,"Oil, Gas and Consumable Fuels",5.28776,5.38904,1885.15154,87.70594,8.39244,1.32097,14.4688,NM


Country Loading

In [21]:
country_lookup_df = conn.execute("""
    SELECT
        m.*,
        t.Country AS Country
    FROM market_cap_df m
    LEFT JOIN ticker_clean_df t
        ON m.cusip = t.Cusip
    WHERE t.country IS NOT NULL
""").fetchdf()

country_lookup_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo,market_cap,Country
0,26883D10,2002-06-04,88817,41004,1.000000,1.000000,27455.0,7.250,0.006944,0.006944,34700.0,7.35,7.270,7.10,7.15,1.990488e+08,United States
1,70469910,2002-06-04,88820,41007,2.541000,2.541000,3336.0,59.000,-0.016667,-0.016667,200.0,61.00,59.000,59.00,59.00,1.968240e+08,United States
2,87938V10,2002-06-04,88825,41010,1.000000,1.000000,697.0,6.150,-0.008064,-0.008064,5300.0,6.18,6.190,6.10,6.12,4.286550e+06,Spain
3,14948120,2002-06-04,88831,41056,0.066667,0.066667,8069.0,-3.385,-0.018841,-0.018841,0.0,3.50,3.500,3.27,3.27,2.731356e+07,United States
4,H2906T10,2002-06-04,88837,41059,2.000000,2.000000,107792.0,23.700,0.006369,0.006369,119753.0,23.70,23.930,23.66,23.35,2.554670e+09,Switzerland
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13792860,13646K10,2023-12-27,89125,41990,1.000000,1.000000,931791.0,79.740,0.003524,0.003524,1460579.0,79.75,80.100,79.74,79.42,7.430101e+10,Canada
13792861,46428746,2023-12-27,89129,37493,1.000000,1.000000,671400.0,75.470,0.005328,0.005328,13103797.0,75.48,75.530,75.47,75.11,5.067056e+10,United States
13792862,69047Q10,2023-12-27,89134,42001,1.000000,1.000000,272902.0,45.000,-0.005085,-0.005085,1519861.0,45.01,45.285,45.00,44.85,1.228059e+10,United States
13792863,H1135610,2023-12-27,89138,42007,1.000000,1.000000,145288.0,102.010,-0.004683,-0.004683,665359.0,102.06,102.320,102.02,101.55,1.482083e+10,United States


Raw Country Factor Returns

In [22]:
country_factors_raw_df = conn.execute("""
WITH lagged AS (
    SELECT
        date,
        permno,
        Country,
        ret,
        LAG(market_cap) OVER (
            PARTITION BY permno
            ORDER BY date
        ) AS lag_mktcap
    FROM country_lookup_df
    WHERE ret IS NOT NULL
      AND market_cap IS NOT NULL
      AND Country IS NOT NULL
),

country_weights AS (
    SELECT
        date,
        Country,
        permno,
        ret,
        lag_mktcap,
        SUM(lag_mktcap) OVER (
            PARTITION BY date, Country
        ) AS country_total_lag_mktcap
    FROM lagged
    WHERE lag_mktcap IS NOT NULL
      AND lag_mktcap > 0
),

country_factor_raw AS (
    SELECT
        date,
        Country,
        SUM(
            (lag_mktcap / NULLIF(country_total_lag_mktcap, 0.0)) * ret
        ) AS country_ret_raw,
        COUNT(*) AS n_stocks
    FROM country_weights
    GROUP BY date, Country
)

SELECT
    date,
    Country,
    country_ret_raw,
    n_stocks
FROM country_factor_raw
ORDER BY date, Country;
""").fetchdf()

country_factors_raw_df

,date,Country,country_ret_raw,n_stocks
0,2002-01-03,0,-0.015183,2
1,2002-01-03,Argentina,-0.010948,4
2,2002-01-03,Australia,0.004639,5
3,2002-01-03,Bahrain,0.015206,1
4,2002-01-03,Belgium,0.016503,2
...,...,...,...,...
286098,2023-12-29,Ukraine,-0.009005,1
286099,2023-12-29,United Arab Emirates,-0.007018,2
286100,2023-12-29,United Kingdom,-0.001015,72
286101,2023-12-29,United States,-0.004402,2989


Pure Country Risk (remove double counted market beta)

In [23]:
country_factors_pure_df = conn.execute("""
WITH joined AS (
    SELECT
        c.date,
        c.Country,
        c.country_ret_raw,
        c.n_stocks AS n_stocks_country,
        m.market_factor_return
    FROM country_factors_raw_df c
    JOIN vw_market_return_df m
      ON c.date = m.date
    WHERE c.country_ret_raw IS NOT NULL
      AND m.market_factor_return IS NOT NULL
),

means AS (
    SELECT
        Country,
        AVG(country_ret_raw)        AS mean_cty,
        AVG(market_factor_return)   AS mean_mkt
    FROM joined
    GROUP BY Country
),

beta AS (
    SELECT
        j.Country,
        SUM((j.country_ret_raw - mn.mean_cty) * (j.market_factor_return - mn.mean_mkt))
          / NULLIF(SUM((j.market_factor_return - mn.mean_mkt) * (j.market_factor_return - mn.mean_mkt)), 0.0)
          AS beta_c_m
    FROM joined j
    JOIN means mn
      ON j.Country = mn.Country
    GROUP BY j.Country
)

SELECT
    j.date,
    j.Country,
    j.country_ret_raw,
    j.market_factor_return,
    b.beta_c_m,
    (j.country_ret_raw - b.beta_c_m * j.market_factor_return) AS country_ret_orth,
    j.n_stocks_country
FROM joined j
JOIN beta b
  ON j.Country = b.Country
ORDER BY j.date, j.Country;
""").fetchdf()

country_factors_pure_df

,date,Country,country_ret_raw,market_factor_return,beta_c_m,country_ret_orth,n_stocks_country
0,2002-01-03,0,-0.015183,0.010148,0.839512,-0.023702,2
1,2002-01-03,Argentina,-0.010948,0.010148,0.801354,-0.019080,4
2,2002-01-03,Australia,0.004639,0.010148,1.288844,-0.008440,5
3,2002-01-03,Bahrain,0.015206,0.010148,1.632464,-0.001360,1
4,2002-01-03,Belgium,0.016503,0.010148,0.787579,0.008510,2
...,...,...,...,...,...,...,...
286098,2023-12-29,Ukraine,-0.009005,-0.004036,1.149197,-0.004366,1
286099,2023-12-29,United Arab Emirates,-0.007018,-0.004036,1.249479,-0.001975,2
286100,2023-12-29,United Kingdom,-0.001015,-0.004036,0.986696,0.002968,72
286101,2023-12-29,United States,-0.004402,-0.004036,0.994894,-0.000386,2989


Industry Loading

In [24]:
industry_lookup_df = conn.execute("""
    SELECT
        m.*,
        t.Industry AS Industry
    FROM market_cap_df m
    LEFT JOIN ticker_clean_df t
        ON m.cusip = t.Cusip
    WHERE t.Industry IS NOT NULL
""").fetchdf()

industry_lookup_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo,market_cap,Industry
0,36720410,2002-01-02,10001,7953,1.50,1.50,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35,11.490,11.19,11.10,2.848260e+07,Gas Utilities
1,46603210,2002-01-02,10026,7976,2.00,2.00,8654.0,24.82,0.015133,0.015133,10544.0,24.84,24.830,24.37,23.46,2.147923e+08,Food Products
2,72913210,2002-01-02,10032,7980,1.00,1.00,41812.0,27.62,0.039910,0.039910,521574.0,27.62,27.690,27.61,26.21,1.154847e+09,"Electronic Equipment, Instruments and Components"
3,00621210,2002-01-02,10065,20023,1.00,1.00,81622.0,14.19,-0.002110,-0.002110,1910800.0,14.23,14.390,14.19,14.15,1.158216e+09,Capital Markets
4,86681020,2002-01-02,10078,8021,0.25,0.25,3248420.0,13.04,0.060163,0.060163,39840954.0,13.04,13.090,13.03,12.50,4.235940e+10,"Technology Hardware, Storage and Peripherals"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13792860,13646K10,2023-12-27,89125,41990,1.00,1.00,931791.0,79.74,0.003524,0.003524,1460579.0,79.75,80.100,79.74,79.42,7.430101e+10,Ground Transportation
13792861,46428746,2023-12-27,89129,37493,1.00,1.00,671400.0,75.47,0.005328,0.005328,13103797.0,75.48,75.530,75.47,75.11,5.067056e+10,Capital Markets
13792862,69047Q10,2023-12-27,89134,42001,1.00,1.00,272902.0,45.00,-0.005085,-0.005085,1519861.0,45.01,45.285,45.00,44.85,1.228059e+10,"Oil, Gas and Consumable Fuels"
13792863,H1135610,2023-12-27,89138,42007,1.00,1.00,145288.0,102.01,-0.004683,-0.004683,665359.0,102.06,102.320,102.02,101.55,1.482083e+10,Food Products


Raw Industry Factor Returns

In [25]:
industry_factors_raw_df = conn.execute("""
WITH lagged AS (
    SELECT
        date,
        permno,
        Industry,
        ret,
        LAG(market_cap) OVER (
            PARTITION BY permno
            ORDER BY date
        ) AS lag_mktcap
    FROM industry_lookup_df
    WHERE ret IS NOT NULL
      AND market_cap IS NOT NULL
      AND Industry IS NOT NULL
),

industry_weights AS (
    SELECT
        date,
        Industry,
        permno,
        ret,
        lag_mktcap,
        SUM(lag_mktcap) OVER (
            PARTITION BY date, Industry
        ) AS industry_total_lag_mktcap
    FROM lagged
    WHERE lag_mktcap IS NOT NULL
      AND lag_mktcap > 0
),

industry_factor_raw AS (
    SELECT
        date,
        Industry,
        SUM(
            (lag_mktcap / NULLIF(industry_total_lag_mktcap, 0.0)) * ret
        ) AS industry_ret_raw,
        COUNT(*) AS n_stocks
    FROM industry_weights
    GROUP BY date, Industry
)

SELECT
    date,
    Industry,
    industry_ret_raw,
    n_stocks
FROM industry_factor_raw
ORDER BY date, Industry;
""").fetchdf()

industry_factors_raw_df


,date,Industry,industry_ret_raw,n_stocks
0,2002-01-03,0,0.023777,9
1,2002-01-03,Aerospace and Defense,-0.002740,18
2,2002-01-03,Air Freight and Logistics,0.019381,6
3,2002-01-03,Automobile Components,0.020017,12
4,2002-01-03,Automobiles,0.018991,9
...,...,...,...,...
415177,2023-12-29,Tobacco,0.003949,3
415178,2023-12-29,Trading Companies and Distributors,-0.006129,29
415179,2023-12-29,Transportation Infrastructure,0.004363,3
415180,2023-12-29,Water Utilities,-0.008730,11


Pure Industry Risk

In [26]:
# Compute Pure Industry Risk in SQL: remove market and country effects
for name in ('industry_factors_raw_df', 'vw_market_return_df', 'country_factors_pure_df'):
    if name not in globals():
        raise RuntimeError(f"Required DataFrame '{name}' not found in notebook namespace")

industry_betas_df = conn.execute("""
WITH joined AS (
    SELECT
        i.date,
        i.Industry,
        i.industry_ret_raw,
        m.market_factor_return
    FROM industry_factors_raw_df i
    JOIN vw_market_return_df m
      ON i.date = m.date
    WHERE i.industry_ret_raw IS NOT NULL
      AND m.market_factor_return IS NOT NULL
),

means_mkt AS (
    SELECT
        Industry,
        AVG(industry_ret_raw)      AS mean_ind,
        AVG(market_factor_return)  AS mean_mkt
    FROM joined
    GROUP BY Industry
),

beta_mkt AS (
    SELECT
        j.Industry,
        SUM((j.industry_ret_raw - mm.mean_ind) * (j.market_factor_return - mm.mean_mkt))
          / NULLIF(SUM((j.market_factor_return - mm.mean_mkt) * (j.market_factor_return - mm.mean_mkt)), 0.0)
          AS beta_market
    FROM joined j
    JOIN means_mkt mm
      ON j.Industry = mm.Industry
    GROUP BY j.Industry
),

resid_mkt AS (
    SELECT
        j.date,
        j.Industry,
        (j.industry_ret_raw - b.beta_market * j.market_factor_return) AS resid_after_market
    FROM joined j
    JOIN beta_mkt b
      ON j.Industry = b.Industry
),

pair_join AS (
    SELECT
        r.Industry,
        r.date,
        r.resid_after_market,
        c.Country,
        c.country_ret_orth
    FROM resid_mkt r
    JOIN country_factors_pure_df c
      ON r.date = c.date
    WHERE r.resid_after_market IS NOT NULL
      AND c.country_ret_orth IS NOT NULL
),

pair_means AS (
    SELECT
        Industry,
        Country,
        AVG(resid_after_market) AS mean_resid,
        AVG(country_ret_orth)   AS mean_cty
    FROM pair_join
    GROUP BY Industry, Country
)

SELECT
    p.Industry,
    p.Country,
    SUM((p.resid_after_market - pm.mean_resid) * (p.country_ret_orth - pm.mean_cty))
      / NULLIF(SUM((p.country_ret_orth - pm.mean_cty) * (p.country_ret_orth - pm.mean_cty)), 0.0)
      AS beta
FROM pair_join p
JOIN pair_means pm
  ON p.Industry = pm.Industry
 AND p.Country = pm.Country
GROUP BY p.Industry, p.Country
ORDER BY p.Industry, p.Country;
""").fetchdf()

industry_factors_pure_df = conn.execute("""
WITH joined AS (
    SELECT
        i.date,
        i.Industry,
        i.industry_ret_raw,
        m.market_factor_return
    FROM industry_factors_raw_df i
    JOIN vw_market_return_df m
      ON i.date = m.date
    WHERE i.industry_ret_raw IS NOT NULL
      AND m.market_factor_return IS NOT NULL
),

means_mkt AS (
    SELECT
        Industry,
        AVG(industry_ret_raw)      AS mean_ind,
        AVG(market_factor_return)  AS mean_mkt
    FROM joined
    GROUP BY Industry
),

beta_mkt AS (
    SELECT
        j.Industry,
        SUM((j.industry_ret_raw - mm.mean_ind) * (j.market_factor_return - mm.mean_mkt))
          / NULLIF(SUM((j.market_factor_return - mm.mean_mkt) * (j.market_factor_return - mm.mean_mkt)), 0.0)
          AS beta_market
    FROM joined j
    JOIN means_mkt mm
      ON j.Industry = mm.Industry
    GROUP BY j.Industry
),

resid_mkt AS (
    SELECT
        j.date,
        j.Industry,
        j.industry_ret_raw,
        j.market_factor_return,
        b.beta_market,
        (j.industry_ret_raw - b.beta_market * j.market_factor_return) AS resid_after_market
    FROM joined j
    JOIN beta_mkt b
      ON j.Industry = b.Industry
),

country_adj AS (
    SELECT
        r.date,
        r.Industry,
        SUM(COALESCE(b.beta, 0.0) * c.country_ret_orth) AS country_adjustment
    FROM resid_mkt r
    LEFT JOIN country_factors_pure_df c
      ON r.date = c.date
    LEFT JOIN industry_betas_df b
      ON r.Industry = b.Industry
     AND c.Country = b.Country
    GROUP BY r.date, r.Industry
)

SELECT
    r.date,
    r.Industry,
    r.industry_ret_raw,
    r.market_factor_return,
    (r.resid_after_market - COALESCE(a.country_adjustment, 0.0)) AS industry_ret_pure,
    r.beta_market
FROM resid_mkt r
LEFT JOIN country_adj a
  ON r.date = a.date
 AND r.Industry = a.Industry
ORDER BY r.date, r.Industry;
""").fetchdf()

conn.register('industry_factors_pure_df', industry_factors_pure_df)
conn.register('industry_betas_df', industry_betas_df)

print('Computed pure industry factors for', industry_factors_pure_df['Industry'].nunique(), 'industries')
industry_factors_pure_df

Computed pure industry factors for 75 industries


,date,Industry,industry_ret_raw,market_factor_return,industry_ret_pure,beta_market
0,2002-01-03,0,0.023777,0.010148,0.026906,1.020212
1,2002-01-03,Aerospace and Defense,-0.002740,0.010148,-0.000669,1.082807
2,2002-01-03,Air Freight and Logistics,0.019381,0.010148,0.009742,0.889526
3,2002-01-03,Automobile Components,0.020017,0.010148,0.010102,1.309775
4,2002-01-03,Automobiles,0.018991,0.010148,-0.023869,1.358354
...,...,...,...,...,...,...
415177,2023-12-29,Tobacco,0.003949,-0.004036,0.003234,0.578877
415178,2023-12-29,Trading Companies and Distributors,-0.006129,-0.004036,-0.003575,1.213170
415179,2023-12-29,Transportation Infrastructure,0.004363,-0.004036,-0.003121,0.941583
415180,2023-12-29,Water Utilities,-0.008730,-0.004036,-0.009053,0.796114


Database Cleanup (full dataset excluding cusips without industry & country)

In [27]:
consolidated_df = conn.execute("""
    SELECT
        m.*,
        t.* EXCLUDE (Cusip)
    FROM market_cap_df m
    INNER JOIN ticker_clean_df t
        ON m.cusip = t.Cusip
    ORDER BY m.date, m.cusip
""").fetchdf()

consolidated_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,...,Country,Industry,EV/EBITDA TTM,EV/EBITDA NTM,EV,EV/UFCF TTM,EV/EBIT TTM,Price/Book TTM,ROIC,Net Debt/EBITDA
0,00036020,2002-01-02,76868,10817,11.390625,11.390625,8666.0,22.00,-0.100940,-0.100940,...,United States,Diversified Consumer Services,0,0,0.00000,0,0,0,7.9809,0.56115
1,00036110,2002-01-02,54594,20000,1.000000,1.000000,26958.0,9.20,0.021088,0.021088,...,United States,Aerospace and Defense,14.97109,13.65208,5644.10256,79.21547,20.41267,2.99475,3.8789,2.59775
2,00037520,2002-01-02,88953,41444,1.038169,1.000000,7388.0,9.89,0.051010,0.051010,...,Switzerland,Electrical Equipment,23.58588,21.08793,130842.14130,127.44298,27.91993,10.06755,11.5772,0.37379
3,00078810,2002-01-02,86828,16482,1.000000,1.000000,12887.0,2.15,-0.088983,-0.088983,...,United States,Capital Markets,0,0,0.00000,0,0,0,0,None
4,00088630,2002-01-02,50906,2902,0.142857,0.142857,792000.0,4.68,0.017391,0.017391,...,United States,Communications Equipment,0,0,0.00000,0,0,0,5.8287,NM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13792860,Y7542C13,2023-12-29,93299,53364,1.000000,1.000000,55000.0,60.80,-0.000164,-0.000164,...,Monaco,"Oil, Gas and Consumable Fuels",7.19426,7.16518,3554.95177,14.07588,11.32865,1.15277,6.9379,NM
13792861,Y8162K20,2023-12-29,92470,52888,1.000000,1.000000,102857.0,21.26,0.003067,0.003067,...,Greece,Marine Transportation,9.66643,6.7052,3698.52273,0,24.76711,1.22447,2.2184,1.82382
13792862,Y8564W10,2023-12-29,81784,30928,1.000000,1.000000,90949.0,7.15,-0.008322,-0.008322,...,Bermuda,"Oil, Gas and Consumable Fuels",4.89815,0,1518.66212,10.56813,8.45962,1.67251,3.8572,NM
13792863,Y8565N30,2023-12-29,92518,52915,1.000000,1.000000,29313.0,49.97,-0.002993,-0.002993,...,Bermuda,"Oil, Gas and Consumable Fuels",5.28776,5.38904,1885.15154,87.70594,8.39244,1.32097,14.4688,NM


Factor Loadings By Equity

In [31]:
from pathlib import Path
import numpy as np
import pandas as pd

# Required tables from prior cells:
# consolidated_df, vw_market_return_df, country_factors_pure_df, industry_factors_pure_df
for name in ["consolidated_df", "vw_market_return_df", "country_factors_pure_df", "industry_factors_pure_df"]:
    if name not in globals():
        raise RuntimeError(f"Missing required DataFrame: {name}")

# ---------------------------
# 1) Build factor matrix by date
# ---------------------------
market_f = (
    vw_market_return_df[["date", "market_factor_return"]]
    .drop_duplicates(subset=["date"])
    .copy()
)

country_pivot = (
    country_factors_pure_df
    .pivot_table(index="date", columns="Country", values="country_ret_orth", aggfunc="mean")
    .fillna(0.0)
)
country_pivot.columns = [f"cty_{str(c).replace(' ', '_')}" for c in country_pivot.columns]

industry_pivot = (
    industry_factors_pure_df
    .pivot_table(index="date", columns="Industry", values="industry_ret_pure", aggfunc="mean")
    .fillna(0.0)
)
industry_pivot.columns = [f"ind_{str(i).replace(' ', '_')}" for i in industry_pivot.columns]

factors_wide = (
    market_f.set_index("date")
    .join(country_pivot, how="left")
    .join(industry_pivot, how="left")
    .fillna(0.0)
    .sort_index()
)

factor_cols = list(factors_wide.columns)

# Use float32 to reduce memory pressure
factors_wide = factors_wide.astype(np.float32)

# ---------------------------
# 2) Prepare equity returns
# ---------------------------
cusip_col = "cusip" if "cusip" in consolidated_df.columns else ("cusip9" if "cusip9" in consolidated_df.columns else None)
if cusip_col is None:
    raise RuntimeError("No equity id column found (expected 'cusip' or 'cusip9').")
if "ret" not in consolidated_df.columns:
    raise RuntimeError("consolidated_df must contain 'ret'.")

eq = consolidated_df[[cusip_col, "date", "ret"]].dropna(subset=[cusip_col, "date", "ret"]).copy()
eq = eq.sort_values([cusip_col, "date"])

# ---------------------------
# 3) Per-equity OLS (memory-safe)
# r_i,t = alpha_i + B_i * F_t + eps_i,t
# ---------------------------
min_obs = max(60, len(factor_cols) + 5)  # keep same logic style, but safe floor

wide_rows = []
n_total = eq[cusip_col].nunique()

for idx, (cid, g) in enumerate(eq.groupby(cusip_col, sort=False), start=1):
    # align factor rows to this equity's dates
    X = factors_wide.reindex(g["date"]).to_numpy(dtype=np.float32)
    y = g["ret"].to_numpy(dtype=np.float32)

    valid = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    if valid.sum() < min_obs:
        continue

    Xv = np.column_stack([np.ones(valid.sum(), dtype=np.float32), X[valid]])
    yv = y[valid]

    coef, *_ = np.linalg.lstsq(Xv, yv, rcond=None)

    row = {
        cusip_col: cid,
        "n_obs": int(valid.sum()),
        "alpha": float(coef[0]),
    }
    for j, f in enumerate(factor_cols, start=1):
        row[f"beta_{f}"] = float(coef[j])

    wide_rows.append(row)

    if idx % 500 == 0:
        print(f"Processed {idx:,}/{n_total:,} equities...")

loadings_wide_df = pd.DataFrame(wide_rows)
print(f"Estimated loadings for {len(loadings_wide_df):,} equities")

# Optional: long format only after wide exists (can be large)
loadings_long_df = loadings_wide_df.melt(
    id_vars=[cusip_col, "n_obs", "alpha"],
    value_vars=[c for c in loadings_wide_df.columns if c.startswith("beta_")],
    var_name="factor",
    value_name="beta"
)
loadings_long_df["factor"] = loadings_long_df["factor"].str.replace("^beta_", "", regex=True)

# Register in DuckDB session if conn exists
if "conn" in globals():
    conn.register("loadings_wide_df", loadings_wide_df)
    conn.register("loadings_long_df", loadings_long_df)

# Save
out_dir = Path.cwd() / "Outputs"
out_dir.mkdir(parents=True, exist_ok=True)

wide_path = out_dir / "cusip_loadings_wide.csv"
long_path = out_dir / "cusip_loadings_long.csv"

loadings_wide_df.to_csv(wide_path, index=False)
loadings_long_df.to_csv(long_path, index=False)

print("Saved wide loadings to Outputs")
print("Saved long loadings to Outputs")


Processed 500/4,611 equities...
Processed 1,000/4,611 equities...
Processed 1,500/4,611 equities...
Processed 2,000/4,611 equities...
Processed 2,500/4,611 equities...
Processed 3,000/4,611 equities...
Processed 3,500/4,611 equities...
Processed 4,000/4,611 equities...
Processed 4,500/4,611 equities...
Estimated loadings for 4,487 equities
Saved wide loadings to Outputs
Saved long loadings to Outputs


In [32]:
loadings_wide_df

,cusip,n_obs,alpha,beta_market_factor_return,beta_cty_0,beta_cty_Angola,beta_cty_Argentina,beta_cty_Australia,beta_cty_Austria,beta_cty_Bahamas,...,beta_ind_Software,beta_ind_Specialized_REITs,beta_ind_Specialty_Retail,"beta_ind_Technology_Hardware,_Storage_and_Peripherals","beta_ind_Textiles,_Apparel_and_Luxury_Goods",beta_ind_Tobacco,beta_ind_Trading_Companies_and_Distributors,beta_ind_Transportation_Infrastructure,beta_ind_Water_Utilities,beta_ind_Wireless_Telecommunication_Services
0,00030710,1251,-0.001031,1.591858,0.086533,3.001488e-13,0.010972,-0.232104,3.382798e-02,-0.005622,...,0.579652,0.396689,-0.067037,0.566373,0.170438,0.101003,0.413954,0.088061,0.129992,-0.365229
1,00036020,5467,0.000561,1.117657,0.014969,-1.146206e-02,-0.000529,-0.037617,1.635336e-02,-0.013313,...,-0.075274,-0.051147,-0.071024,-0.033729,-0.025886,-0.035696,0.122947,-0.002096,0.074751,0.023253
2,00036110,5480,0.000248,1.424868,0.101628,-5.290705e-02,0.051515,-0.090460,4.169829e-02,0.211915,...,0.148776,-0.186651,0.013963,0.085896,0.045431,0.019742,0.257999,0.013256,0.143400,0.048510
3,00037520,5271,0.000197,1.240770,-0.017681,9.105378e-03,-0.020323,0.025917,-4.924349e-03,0.012223,...,0.032032,0.095389,0.127655,0.076128,-0.043329,-0.097704,0.008223,0.017087,0.018281,0.020252
4,00039J10,308,0.000201,0.027946,-0.009808,7.809725e-15,-0.000190,0.006171,-1.402524e-15,0.001212,...,-0.001831,0.001207,0.020480,0.010656,-0.007481,-0.001575,0.008508,0.002960,0.007236,-0.003690
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4482,Y8162K20,3912,-0.000160,1.038064,-0.017666,-1.073308e-13,0.043826,0.072327,2.669556e-02,0.034955,...,0.074763,-0.069332,0.009435,0.128873,-0.024960,-0.019447,-0.054536,-0.000858,0.026770,0.055258
4483,Y8564W10,5447,-0.000102,1.327781,-0.044169,2.361842e-02,0.041564,0.057487,3.303815e-03,-0.006390,...,0.090988,0.153163,0.045452,0.071997,0.002223,0.038453,0.061421,-0.014983,-0.113049,-0.004367
4484,Y8565J10,3198,0.000159,1.010868,-0.121474,1.124587e-12,0.086373,-0.079854,6.271948e-02,-0.025091,...,0.210677,0.049347,0.298042,0.347342,-0.067813,-0.002346,0.136548,0.018585,-0.152612,0.028970
4485,Y8565N30,3869,0.000281,1.224554,-0.024291,-8.899825e-13,-0.037998,0.063908,5.325835e-03,0.041608,...,0.132823,0.116010,-0.064391,0.112586,-0.020969,0.031718,0.139380,0.012102,0.013716,0.052211


In [33]:
loadings_long_df

,cusip,n_obs,alpha,factor,beta
0,00030710,1251,-0.001031,market_factor_return,1.591858
1,00036020,5467,0.000561,market_factor_return,1.117657
2,00036110,5480,0.000248,market_factor_return,1.424868
3,00037520,5271,0.000197,market_factor_return,1.240770
4,00039J10,308,0.000201,market_factor_return,0.027946
...,...,...,...,...,...
628175,Y8162K20,3912,-0.000160,ind_Wireless_Telecommunication_Services,0.055258
628176,Y8564W10,5447,-0.000102,ind_Wireless_Telecommunication_Services,-0.004367
628177,Y8565J10,3198,0.000159,ind_Wireless_Telecommunication_Services,0.028970
628178,Y8565N30,3869,0.000281,ind_Wireless_Telecommunication_Services,0.052211
